In [0]:
# VoeBem Analytics AI
# Camada Bronze - Dados de Referência
# Fonte: ANAC - Dados Abertos

CATALOG = "voebem"
SCHEMA_BRONZE = "bronze"

VOLUME_REFERENCIAS = "/Volumes/voebem/bronze/referências"

arquivos = dbutils.fs.ls(VOLUME_REFERENCIAS)

print(f"Arquivos encontrados: {len(arquivos)}")

for arquivo in arquivos:
    print(arquivo.name)

In [0]:
# Inspeção rápida dos 3 arquivos de referência
# Os arquivos utilizam codificação Latin/ISO-8859-1

for arquivo in arquivos:
    print("=" * 80)
    print(f"ARQUIVO: {arquivo.name}")

    df_teste = (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "ISO-8859-1")
        .option("inferSchema", "false")
        .csv(arquivo.path)
    )

    print(f"Registros: {df_teste.count():,}")
    print(f"Colunas: {len(df_teste.columns)}")
    print("Nomes:", df_teste.columns)
    print()

In [0]:
# Leitura correta dos três arquivos de referência
# A primeira linha ("Atualizado em...") é ignorada.

def ler_referencia(nome_arquivo):
    return (
        spark.read
        .option("header", "true")
        .option("sep", ";")
        .option("quote", '"')
        .option("encoding", "ISO-8859-1")
        .option("inferSchema", "false")
        .option("skipRows", 1)
        .csv(f"{VOLUME_REFERENCIAS}/{nome_arquivo}")
    )

df_aerodromos = ler_referencia("AerodromosPublicos.csv")

df_empresas_nacionais = ler_referencia(
    "pda_empresas_aereas_nacionais.csv"
)

df_empresas_estrangeiras = ler_referencia(
    "pda_empresas_aereas_estrangeiros.csv"
)

for nome, df in [
    ("Aeródromos", df_aerodromos),
    ("Empresas nacionais", df_empresas_nacionais),
    ("Empresas estrangeiras", df_empresas_estrangeiras)
]:
    print("=" * 70)
    print(nome)
    print(f"Registros: {df.count():,}")
    print(f"Colunas: {len(df.columns)}")
    print(f"Nomes: {df.columns}")

In [0]:
import re
import unicodedata
from pyspark.sql.functions import current_timestamp, lit

# ---------------------------------------------------------
# Função para normalizar nomes de colunas
# ---------------------------------------------------------

def normalizar_nome_coluna(nome):
    nome = unicodedata.normalize("NFKD", nome)
    nome = nome.encode("ascii", "ignore").decode("ascii")
    nome = nome.lower().strip()

    nome = re.sub(r"[^a-z0-9]+", "_", nome)
    nome = re.sub(r"_+", "_", nome)

    return nome.strip("_")


def preparar_bronze(df, fonte):
    
    # Normaliza nomes das colunas
    for coluna in df.columns:
        df = df.withColumnRenamed(
            coluna,
            normalizar_nome_coluna(coluna)
        )

    # Metadados de ingestão
    df = (
        df
        .withColumn("_fonte", lit(fonte))
        .withColumn("_data_ingestao", current_timestamp())
    )

    return df


# ---------------------------------------------------------
# Preparação dos DataFrames
# ---------------------------------------------------------

df_aerodromos_bronze = preparar_bronze(
    df_aerodromos,
    "ANAC - Aerodromos Publicos"
)

df_empresas_nacionais_bronze = preparar_bronze(
    df_empresas_nacionais,
    "ANAC - Empresas Aereas Nacionais"
)

df_empresas_estrangeiras_bronze = preparar_bronze(
    df_empresas_estrangeiras,
    "ANAC - Empresas Aereas Estrangeiras"
)


# ---------------------------------------------------------
# Persistência em Delta / Unity Catalog
# ---------------------------------------------------------

tabelas = [
    (
        df_aerodromos_bronze,
        f"{CATALOG}.{SCHEMA_BRONZE}.aerodromos"
    ),
    (
        df_empresas_nacionais_bronze,
        f"{CATALOG}.{SCHEMA_BRONZE}.empresas_nacionais"
    ),
    (
        df_empresas_estrangeiras_bronze,
        f"{CATALOG}.{SCHEMA_BRONZE}.empresas_estrangeiras"
    )
]


for df, tabela in tabelas:
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela)
    )

    print(f"OK -> {tabela}: {df.count():,} registros")